In [18]:
import math
import numpy as np

In [19]:
# # Airy disk phenomena:

# def divergence_deg(wavelength_um, aperture_mm, M2=1.0):
#     """Diffraction limited full angle divergence in degrees."""
    
#     lam = wavelength_um * 1e-6
#     D = aperture_mm * 1e-3
    
#     theta_rad = 1.22 * lam / D
#     theta_deg = math.degrees(theta_rad)

#     return theta_deg

# # Examples
# for wavelength in [1.55, 4, 8, 10]:
#     print(f"\n{wavelength} µm")

#     for aperture in [25, 50, 100]:
#         div = divergence_deg(wavelength, aperture)
#         print(f"  {aperture:3d} mm aperture: {div:.5f} deg")


In [25]:
def coll_beam_divergence_deg(wavelength_um, beam_diameter_mm, M2=1.0):
    """Diffraction limited half-angle divergence in degrees for a Gaussian beam.
        Assuming lens aperture larger than beam diameter. M2 is the beam quality factor (1 for ideal Gaussian)."""
    
    
    lam = wavelength_um * 1e-6
    D = beam_diameter_mm * 1e-3
    w0 = D / 2  # Beam waist radius for a collimated beam
    
    theta_rad = M2 * lam / (math.pi * w0)  # w0 is the beam waist radius, which can be approximated as D/2 for a collimated beam
    theta_deg = math.degrees(theta_rad)

    return theta_deg

# Examples
for wavelength in [1.55, 4, 8, 10]:
    print(f"\n{wavelength} µm")

    for beam_waist in [25, 50]:
        beam_diameter = 2 * beam_waist  # Convert waist radius to diameter
        div = coll_beam_divergence_deg(wavelength, beam_diameter)
        print(f"  {beam_waist:3d} mm beam waist, half-angle divergence: {div:.4f} deg")


1.55 µm
   25 mm beam waist, half-angle divergence: 0.0011 deg
   50 mm beam waist, half-angle divergence: 0.0006 deg

4 µm
   25 mm beam waist, half-angle divergence: 0.0029 deg
   50 mm beam waist, half-angle divergence: 0.0015 deg

8 µm
   25 mm beam waist, half-angle divergence: 0.0058 deg
   50 mm beam waist, half-angle divergence: 0.0029 deg

10 µm
   25 mm beam waist, half-angle divergence: 0.0073 deg
   50 mm beam waist, half-angle divergence: 0.0036 deg


In [4]:


def fso_geometric_loss(distance_m, divergence_deg, receiver_diameter_m):
    theta = math.radians(divergence_deg)

    beam_diameter = theta * distance_m  # small-angle approximation

    if beam_diameter <= receiver_diameter_m:
        return 0.0  # all power collected

    loss_db = 20 * math.log10(beam_diameter / receiver_diameter_m)
    return loss_db

# Example:
distance = 1000          # 1 km
divergence_deg = 0.01    # 0.01 degree
receiver_diameter = 0.1  # 10 cm

loss = fso_geometric_loss(distance,
                          divergence_deg,
                          receiver_diameter)

print(f"Geometric loss = {loss:.2f} dB")

Geometric loss = 4.84 dB


In [5]:
def fso_loss(distance_m,
             wavelength_um,
             tx_aperture_mm,
             rx_aperture_mm,
             M2=1.5):

    lam = wavelength_um * 1e-6
    tx = tx_aperture_mm * 1e-3
    rx = rx_aperture_mm * 1e-3

    # divergence
    theta = M2 * 1.22 * lam / tx

    # beam diameter at receiver
    beam_diameter = theta * distance_m

    # collected fraction
    if beam_diameter <= rx:
        collection = 1.0
    else:
        collection = (rx / beam_diameter)**2

    loss_db = -10 * math.log10(collection)

    return {
        "divergence_deg": math.degrees(theta),
        "beam_diameter_m": beam_diameter,
        "loss_db": loss_db
    }


result = fso_loss(
    distance_m=10_000,
    wavelength_um=10,
    tx_aperture_mm=100,
    rx_aperture_mm=100,
    M2=1.5
)

print(result)

{'divergence_deg': 0.010485127650894063, 'beam_diameter_m': 1.8299999999999998, 'loss_db': 25.24902179460859}
